In [ ]:
from pathlib import Path
import subprocess
from faster_whisper import WhisperModel

In [ ]:
def detect_language_from_first_5_minutes(
    video_path: str,
    sample_audio_path: str = "language_sample.wav",
    ffmpeg_path: str = "ffmpeg",
    model_size: str = "medium",
    device: str = "cpu",
    compute_type: str = "int8",
    sample_seconds: int = 300
):
    """
    Extract the first 5 minutes of audio from a video and detect the language.
    """

    video_file = Path(video_path)
    sample_file = Path(sample_audio_path)

    if not video_file.exists():
        raise FileNotFoundError(f"video file not found: {video_file}")

    # extract first 5 minutes as mono 16k wav
    cmd = [
        ffmpeg_path,
        "-y",
        "-i", str(video_file),
        "-t", str(sample_seconds),
        "-vn",
        "-ac", "1",
        "-ar", "16000",
        str(sample_file)
    ]

    print("extracting sample audio...")
    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("ffmpeg failed while extracting sample audio")

    print(f"sample created: {sample_file}")

    # load model
    model = WhisperModel(model_size, device=device, compute_type=compute_type)

    # transcribe a short sample and let whisper detect language
    segments, info = model.transcribe(
        str(sample_file),
        language=None,
        beam_size=5
    )

    # force evaluation of segments
    preview_segments = list(segments)

    print(f"detected language: {info.language}")
    print(f"language probability: {info.language_probability:.3f}")

    print("\npreview of detected speech:")
    for i, segment in enumerate(preview_segments[:10]):
        print(f"[{segment.start:.2f} -> {segment.end:.2f}] {segment.text}")

    return info.language, info.language_probability, preview_segments

In [ ]:
lang, prob, segs = detect_language_from_first_5_minutes(
    video_path="My_Video.mp4",
    ffmpeg_path="ffmpeg",
    model_size="medium"
)